# RSO-901 Glycol Set Points — Command Response

How quickly, and how reliably, do the two EAS-controlled glycol chillers (Chiller 01 and Chiller 02)
follow the set points that EAS commands? This compares every `HVAC.configChiller` command with the
chiller's own reported set point over an arbitrary `day_obs` range (set `start_day_obs` and
`end_day_obs` below, both integers `YYYYMMDD`; the window runs from 12:00 UTC on the first `day_obs`
through 11:59:59 UTC the day after the last one).

It answers three questions:

1. **Latency** — when EAS commands a *new* value, how long until the chiller reports it, and how often
   does it never do so?
2. **Time mismatched** — EAS re-sends every value each 60 s; how much of the time is the chiller's
   reported set point not the commanded one? This is the total time the system was not behaving as
   expected, and it is measured from the data, not from how long anyone took to recover.
3. **Size** — how far off is the chiller during a mismatch? Most mismatches are a fraction of a degree
   and do not matter, so the size section picks out the *large* ones (`LARGE_MISMATCH_C` and above) and
   shows how big they are and how much time they account for.

**Why these topics.** `command_configChiller` is what EAS asked for. `logevent_chillerConfiguration`
looks like a record of the commanded value, but the HVAC CSC builds it from the chiller's *telemetry*
(`_send_config_event` in `ts_hvac`), and only while the chiller is switched on. It is therefore the
chiller's **readback**, written only when the readback changes, which makes it cheap to query over many
months. `logevent_deviceEnabled` says when each chiller was switched off, so readbacks are not judged
while the chiller is off.

Method notes: the chiller truncates set points toward zero to 0.1 °C (commanded 6.18 reads back 6.1,
and −1.97 reads back −1.9), so a readback "matches" a command when it is within 0.1 °C of it. Commands are only judged
once `RESPONSE_WINDOW_S` seconds have passed since the commanded value last changed.

See `ai_notes.md` in this folder for the topic survey and field reference. This is the command-side
companion to the *Date Range* notebook.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os

import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The getDayObs* helpers live in dateTime; importing them from efdUtils is
# deprecated and those shims are removed after w_2026_01.
from lsst.summit.utils.dateTime import getDayObsStartTime, getDayObsEndTime
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.ts.xml.enums.HVAC import DeviceId


def show_figure(fig):
    """Show a figure once.

    On the inline backend the figure is closed afterwards, otherwise it is drawn again at the end of
    the next figure cell. An interactive (ipympl, ``%matplotlib widget``) figure must stay open: closing
    it destroys its widget model and the notebook reports "model not found".
    """
    plt.show()
    if "ipympl" not in matplotlib.get_backend():
        plt.close(fig)

## Parameters

In [ ]:
# Inclusive day_obs range, as integers YYYYMMDD. The observatory day rolls over at
# 12:00 UTC (UTC-12 convention), so the window is 12:00 UTC on start_day_obs through
# 11:59:59 UTC the day after end_day_obs.
start_day_obs = 20260101
end_day_obs = 20260720  # storm night

# device_id (DeviceId_chiller enum) -> name. Only the two chillers EAS commands.
CHILLERS = {
    101: "chiller01",
    102: "chiller02",
}

COMMAND_TOPIC = "lsst.sal.HVAC.command_configChiller"  # what EAS asked for
READBACK_TOPIC = "lsst.sal.HVAC.logevent_chillerConfiguration"  # what the chiller reports
ENABLED_TOPIC = "lsst.sal.HVAC.logevent_deviceEnabled"  # bitmask of switched-on devices

# A changed command counts as answered if the readback matches within this many seconds, and a
# command is only judged for mismatch once its value is at least this old.
RESPONSE_WINDOW_S = 60.0

# The chiller truncates toward zero to 0.1 degC (6.18 reads 6.1, -1.97 reads -1.9), so a readback
# matches if it is within this of the command, in either direction.
READBACK_TOLERANCE = 0.1

# A mismatch counts as "large" when the readback is at least this far from the command (degC, either
# direction). Smaller mismatches still appear in the all-mismatch numbers but are not of interest in the
# size section.
LARGE_MISMATCH_C = 0.5

# Each judged command stands for the time until the next command, capped at this many seconds so a
# gap in the EAS commands (for example EAS being down) is not counted as mismatch time.
MAX_COMMAND_GAP_S = 120.0

# Query the EFD this many day_obs at a time, so a multi-month range does not time out.
CHUNK_DAYS = 10

# Curated CSV of catastrophic glycol / camera-cooling events, overlaid on the mismatch timeline.
EVENTS_CSV = "glycol_catastrophic_faults.csv"

# Position of each chiller in the deviceEnabled bitmask: the HVAC CSC numbers devices by their
# order in the DeviceId enum.
ENABLED_BIT = {dev: list(DeviceId).index(DeviceId(dev)) for dev in CHILLERS}

In [ ]:
if end_day_obs < start_day_obs:
    raise ValueError(f"end_day_obs {end_day_obs} < start_day_obs {start_day_obs}")

efd_client = makeEfdClient()

begin = getDayObsStartTime(start_day_obs)  # 12:00 UTC on start_day_obs
end = getDayObsEndTime(end_day_obs)  # 11:59:59 UTC the day after end_day_obs

print(f"day_obs {start_day_obs}\u2013{end_day_obs}: {begin.isot} -> {end.isot} (UTC)")

## Query the commands, readbacks and enabled mask

Every query is split into `CHUNK_DAYS`-day pieces. The commands are the dense part: EAS sends one per
chiller per minute, about 580 000 rows over six months. The readback and enabled events are sparse, so
they are cheap.

In [ ]:
days = pd.date_range(str(start_day_obs), str(end_day_obs), freq="D")
chunks = [days[i : i + CHUNK_DAYS] for i in range(0, len(days), CHUNK_DAYS)]


def day_obs_of(timestamp):
    return int(timestamp.strftime("%Y%m%d"))


def query_chunked(topic, **kwargs):
    frames = []
    for chunk in chunks:
        part = getEfdData(
            efd_client,
            topic,
            begin=getDayObsStartTime(day_obs_of(chunk[0])),
            end=getDayObsEndTime(day_obs_of(chunk[-1])),
            **kwargs,
        )
        if len(part):
            frames.append(part)
    return pd.concat(frames).sort_index() if frames else pd.DataFrame()


def to_naive_utc(frame):
    # Drop tz (values are UTC) so everything compares against the naive window and events below.
    if len(frame) and frame.index.tz is not None:
        frame = frame.copy()
        frame.index = frame.index.tz_convert("UTC").tz_localize(None)
    return frame


raw_commands = to_naive_utc(
    query_chunked(COMMAND_TOPIC, columns=["activeSetpoint", "device_id", "private_identity"])
)
raw_readbacks = to_naive_utc(query_chunked(READBACK_TOPIC, columns=["activeSetpoint", "device_id"]))
raw_enabled = to_naive_utc(query_chunked(ENABLED_TOPIC, columns=["device_ids"]))

print(f"{len(raw_commands)} command(s), {len(raw_readbacks)} readback event(s), "
      f"{len(raw_enabled)} deviceEnabled event(s)")
raw_commands.head() if len(raw_commands) else raw_commands

## Line up commands with readbacks

For each chiller this attaches to every command the readback in force when it arrived, whether the
chiller was switched on, and how long ago the commanded value last changed. A command that merely
repeats the previous value is `changed = False`; the first command in the range is not a change either,
because there is nothing to compare with.

`deviceEnabled` is an event, so the state at the start of the range is unknown and the chiller is
assumed to be on until the first event says otherwise.

In [ ]:
def matches(readback, command):
    """True where the readback is the command as the chiller reports it (truncated to 0.1 degC)."""
    # The 1e-6 absorbs the float32 noise in values such as 5.19999980926.
    return np.abs(readback - command) <= READBACK_TOLERANCE + 1e-6


def asof(index, series):
    """Value of a step series in force at each timestamp of ``index`` (NaN before its first row)."""
    series = series.sort_index()
    if not len(series):
        return np.full(len(index), np.nan)
    pos = series.index.searchsorted(index, side="right") - 1
    values = series.to_numpy(dtype=float)[np.clip(pos, 0, None)]
    return np.where(pos >= 0, values, np.nan)


commands, readbacks, enabled_state = {}, {}, {}
for dev in CHILLERS:
    readbacks[dev] = raw_readbacks.loc[raw_readbacks["device_id"] == dev, "activeSetpoint"].sort_index()

    if len(raw_enabled):
        bits = raw_enabled["device_ids"].astype("int64").to_numpy() >> ENABLED_BIT[dev] & 1
        enabled_state[dev] = pd.Series(bits, index=raw_enabled.index).sort_index()
    else:
        enabled_state[dev] = pd.Series(dtype=float)

    c = raw_commands.loc[raw_commands["device_id"] == dev].sort_index().copy()
    c["changed"] = c["activeSetpoint"].diff().abs().gt(1e-6)  # first row: NaN diff -> False
    c["readback"] = asof(c.index, readbacks[dev])
    c["enabled"] = np.nan_to_num(asof(c.index, enabled_state[dev]), nan=1.0).astype(bool)
    c["matched"] = matches(c["readback"], c["activeSetpoint"])
    c["error"] = c["readback"] - c["activeSetpoint"]  # negative: chiller colder than commanded

    # Age of the commanded value: seconds since it last changed (or since the first command).
    stamp = c.index.to_series()
    last_change = stamp.where(c["changed"] | (stamp == stamp.iloc[0])).ffill() if len(c) else stamp
    c["age_s"] = (stamp - last_change).dt.total_seconds()
    # Time this command stands for: until the next command, capped; the last one counts as one minute.
    gap_s = (stamp.shift(-1) - stamp).dt.total_seconds().clip(upper=MAX_COMMAND_GAP_S)
    c["interval_min"] = (gap_s / 60).fillna(1.0)
    commands[dev] = c

for dev, name in CHILLERS.items():
    c = commands[dev]
    print(f"{name}: {len(c)} commands, {int(c['changed'].sum())} changed, "
          f"{len(readbacks[dev])} readback events, "
          f"{int((~c['enabled']).sum())} commands sent while the chiller was off")

if len(raw_commands) and "private_identity" in raw_commands:
    print("\nCommand senders:")
    print(raw_commands["private_identity"].value_counts())

## 1. Response to a new command

For each command whose value differs from the previous one, find the first readback event within
`RESPONSE_WINDOW_S` that matches it. Each command gets one status:

| Status | Meaning |
|---|---|
| `responded` | The readback matched within the window; `latency_s` is the delay. |
| `no_response` | The window passed without a matching readback. |
| `already_matched` | The readback already matched when the command arrived (for example a 0.05 °C change that truncates to the same value), so no event is expected. |
| `chiller_off` | The chiller was switched off, so no readback is expected. |

In [ ]:
window = pd.Timedelta(seconds=RESPONSE_WINDOW_S)


def first_match_latency(series, t, target):
    lo = series.index.searchsorted(t, side="left")
    hi = series.index.searchsorted(t + window, side="right")
    for stamp, value in zip(series.index[lo:hi], series.iloc[lo:hi]):
        if matches(value, target):
            return (stamp - t).total_seconds()
    return np.nan


rows = []
for dev, name in CHILLERS.items():
    c = commands[dev]
    previous = c["activeSetpoint"].shift()
    for t, row in c[c["changed"]].iterrows():
        if not row["enabled"]:
            status, latency = "chiller_off", np.nan
        elif row["matched"]:
            status, latency = "already_matched", np.nan
        else:
            latency = first_match_latency(readbacks[dev], t, row["activeSetpoint"])
            status = "responded" if not np.isnan(latency) else "no_response"
        rows.append(
            dict(
                time=t,
                chiller=name,
                old_command=previous[t],
                command=row["activeSetpoint"],
                readback_before=row["readback"],
                status=status,
                latency_s=latency,
            )
        )

latency = pd.DataFrame(rows)
if len(latency):
    print(latency.groupby(["chiller", "status"]).size().unstack(fill_value=0))
    print()
    print(latency.groupby("chiller")["latency_s"].describe(percentiles=[0.5, 0.9, 0.99]).round(1))
    display(latency[latency["status"] == "no_response"])
else:
    print("No changed commands in this range.")

## Figure — response latency

Left: distribution of the delay between a new command and the matching readback. Right: the same
delays through time, with unanswered commands drawn as red crosses at the top of the window.

In [ ]:
stem = f"chiller_command_response_{start_day_obs}_{end_day_obs}"
os.makedirs("plots", exist_ok=True)

if len(latency):
    fig, (ax_hist, ax_time) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw=dict(width_ratios=[1, 2]))
    bins = np.linspace(0, RESPONSE_WINDOW_S, int(RESPONSE_WINDOW_S) + 1)

    for name in CHILLERS.values():
        g = latency[latency["chiller"] == name]
        ok = g[g["status"] == "responded"]
        ax_hist.hist(ok["latency_s"], bins=bins, alpha=0.6, label=f"{name} (n={len(ok)})")
        ax_time.plot(ok["time"], ok["latency_s"], "o", ms=3, alpha=0.7, label=name)
        lost = g[g["status"] == "no_response"]
        ax_time.plot(lost["time"], np.full(len(lost), RESPONSE_WINDOW_S), "x", color="#E5484D", ms=8)

    ax_hist.set_xlabel("Latency to matching readback [s]")
    ax_hist.set_ylabel("Commands")
    ax_hist.legend()
    ax_time.set_ylabel("Latency [s]")
    ax_time.set_xlabel("Time [UTC]")
    ax_time.set_ylim(0, RESPONSE_WINDOW_S * 1.05)
    ax_time.legend()
    for ax in (ax_hist, ax_time):
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"Chiller response to a new set point — day_obs {start_day_obs}\u2013{end_day_obs}")
    fig.tight_layout()
    fig.savefig(f"plots/{stem}_latency.png", dpi=150, bbox_inches="tight")
    show_figure(fig)
else:
    print("Nothing to plot.")

## 2. Time spent mismatched

Every command (new or repeated) is judged against the readback in force when it arrived, but only when

* the chiller was switched on,
* a readback exists, and
* the commanded value is older than `RESPONSE_WINDOW_S`, so a chiller still on its way to a new value
  is not counted.

Each judged command stands for the time until the next command (capped at `MAX_COMMAND_GAP_S`, so a gap
in the EAS commands is not counted). Adding those intervals up for the mismatching commands gives the
**time the chiller was not at its commanded set point**. That total is what this section reports; how
long it took anyone to notice and recover is not a property of the system, so it is not measured here.

A **mismatch period** is a run of consecutive judged commands that all mismatch. Periods only serve to
locate when it happened and to give the worst error in each run.

In [ ]:
def find_periods(judged, flag, name, all_commands):
    """Runs of consecutive judged commands where ``flag`` is True, one row per run."""
    rows = []
    run = (judged[flag] != judged[flag].shift()).cumsum()
    for _, g in judged[judged[flag]].groupby(run[judged[flag]]):
        start, stop = g.index[0], g.index[-1]
        worst = g["error"].abs().idxmax()
        rows.append(
            dict(
                chiller=name,
                start=start,
                end=stop,
                mismatch_min=g["interval_min"].sum(),
                n_commands=len(g),
                command_first=g["activeSetpoint"].iloc[0],
                command_last=g["activeSetpoint"].iloc[-1],
                readback_first=g["readback"].iloc[0],
                readback_last=g["readback"].iloc[-1],
                max_abs_error_c=g["error"].abs().max(),
                worst_error_c=g.loc[worst, "error"],
                off_during=bool((~all_commands.loc[start:stop, "enabled"]).any()),
            )
        )
    return rows


period_rows, large_rows, summary_rows, judged_parts = [], [], [], []
for dev, name in CHILLERS.items():
    c = commands[dev]
    judged = c[c["enabled"] & c["readback"].notna() & (c["age_s"] > RESPONSE_WINDOW_S)].copy()
    judged["mismatch"] = ~judged["matched"]
    judged["large"] = judged["mismatch"] & (judged["error"].abs() >= LARGE_MISMATCH_C)
    judged_parts.append(judged.assign(chiller=name))

    judged_h = judged["interval_min"].sum() / 60
    mismatch_h = judged.loc[judged["mismatch"], "interval_min"].sum() / 60
    large_h = judged.loc[judged["large"], "interval_min"].sum() / 60
    summary_rows.append(
        dict(
            chiller=name,
            judged_hours=judged_h,
            mismatch_hours=mismatch_h,
            mismatch_pct=100 * mismatch_h / judged_h if judged_h else np.nan,
            large_hours=large_h,
            large_pct=100 * large_h / judged_h if judged_h else np.nan,
        )
    )
    period_rows += find_periods(judged, "mismatch", name, c)
    large_rows += find_periods(judged, "large", name, c)

judged_all = pd.concat(judged_parts)
periods = pd.DataFrame(period_rows)
large_periods = pd.DataFrame(large_rows)

summary = pd.DataFrame(summary_rows).set_index("chiller")
for label, per in (("periods", periods), ("large_periods", large_periods)):
    summary[label] = per.groupby("chiller").size() if len(per) else 0
summary = summary.fillna(0)
print(f"Time with the readback off the command by more than {READBACK_TOLERANCE} degC (mismatch) and by "
      f"{LARGE_MISMATCH_C} degC or more (large), out of the time judged:")
display(summary.round(2))

## Figure — time in mismatch

Hours per UTC day that each chiller spent mismatched: coloured for large mismatches
(`≥ LARGE_MISMATCH_C`), grey for smaller ones. Dashed vertical lines are the catastrophic cooling events
from `glycol_catastrophic_faults.csv` (red for realised faults, amber for near-misses), tagged `#N` as in
the *Date Range* notebook.

In [ ]:
EVENT_COLORS = {"actual": "#E5484D", "near_miss": "#F5A623"}
CHILLER_COLORS = {"chiller01": "#00C2C2", "chiller02": "#FF7A00"}

events = pd.read_csv(EVENTS_CSV)
events["t"] = pd.to_datetime(events["fault_start_timestamp_utc"], utc=True).dt.tz_localize(None)
win_lo, win_hi = begin.utc.datetime, end.utc.datetime
events_win = events[(events["t"] >= win_lo) & (events["t"] <= win_hi)].sort_values("t")
print(f"{len(events_win)} catastrophic event(s) within the window")

fig, axes = plt.subplots(len(CHILLERS), 1, figsize=(14, 2.6 * len(CHILLERS)), sharex=True)
for ax, (name, color) in zip(np.atleast_1d(axes), CHILLER_COLORS.items()):
    g = judged_all[judged_all["chiller"] == name]
    small_h = (g["interval_min"] * (g["mismatch"] & ~g["large"])).resample("D").sum() / 60
    large_h = (g["interval_min"] * g["large"]).resample("D").sum() / 60
    days_num = mdates.date2num(small_h.index)
    # align="edge": each bar spans its own UTC day rather than straddling midnight.
    ax.bar(days_num, small_h.to_numpy(), width=0.9, align="edge", color="#b8b7b1",
           label=f"< {LARGE_MISMATCH_C} \u00b0C")
    ax.bar(days_num, large_h.to_numpy(), width=0.9, align="edge", bottom=small_h.to_numpy(), color=color,
           label=f"\u2265 {LARGE_MISMATCH_C} \u00b0C")
    ax.set_ylim(bottom=0)
    for _, ev in events_win.iterrows():
        ev_color = EVENT_COLORS.get(ev["outcome"], "#888888")
        ax.axvline(ev["t"], color=ev_color, ls="--", lw=1.2)
        ax.annotate(f"#{int(ev['event_id'])}", (ev["t"], 1.0), xycoords=("data", "axes fraction"),
                    color=ev_color, fontsize=8, ha="center", va="bottom", annotation_clip=False)
    ax.set_ylabel("Hours / day")
    ax.set_title(f"{name}: time with readback \u2260 command", loc="left", fontsize=10)
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
ax.xaxis_date()
ax.set_xlim(win_lo, win_hi)
locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
ax.set_xlabel("Time [UTC]")
fig.tight_layout()
fig.savefig(f"plots/{stem}_mismatch_time.png", dpi=150, bbox_inches="tight")
show_figure(fig)

## 3. How large are the mismatches?

The error is `readback − command` (negative means the chiller is colder than commanded). The first table
is the **hours** spent at each error size, so it shows how much of the mismatch time is the small
(sub-0.5 °C) kind and how much is large. Then the large mismatch periods are listed, in time order, with
the time actually spent mismatched in each and the worst error.

In [ ]:
size_bins = [READBACK_TOLERANCE, 0.25, 0.5, 1, 2, 5, np.inf]
size_labels = ["0.1\u20130.25", "0.25\u20130.5", "0.5\u20131", "1\u20132", "2\u20135", ">5"]

mismatched = judged_all[judged_all["mismatch"]].copy()
mismatched["size_c"] = pd.cut(mismatched["error"].abs(), size_bins, labels=size_labels)
mismatched["hours"] = mismatched["interval_min"] / 60
print("Hours mismatched, by |readback - command| [degC]:")
print(mismatched.groupby(["chiller", "size_c"], observed=False)["hours"].sum().unstack(fill_value=0).round(2))
print()

for name in CHILLERS.values():
    m = mismatched[mismatched["chiller"] == name]
    if not len(m):
        print(f"{name}: no mismatches")
        continue
    big_h = m.loc[m["large"], "hours"].sum()
    print(f"{name}: {m['hours'].sum():.1f} h mismatched, {big_h:.1f} h of it large "
          f"({100 * big_h / m['hours'].sum():.0f}%); median |error| {m['error'].abs().median():.2f} degC, "
          f"largest {m['error'].abs().max():.2f} degC")

print()
if len(large_periods):
    print(f"{len(large_periods)} large mismatch period(s):")
    display(large_periods.sort_values("start").reset_index(drop=True))
else:
    print(f"No mismatch reached {LARGE_MISMATCH_C} degC in this range.")

## Figure — all judged commands, and how far off they are

Unlike the mismatch tables, these use **every judged command**, matching or not, so the scatter shows
where the chillers normally sit. **Left:** readback against command, as a density (log colour) with the
mismatching commands overlaid as points and the `±LARGE_MISMATCH_C` band shaded. **Right:** the error
through time, matching commands in grey and mismatches in colour.

In [ ]:
if len(judged_all):
    fig, (ax_xy, ax_t) = plt.subplots(1, 2, figsize=(14, 5))

    lims = [judged_all[["activeSetpoint", "readback"]].min().min() - 1,
            judged_all[["activeSetpoint", "readback"]].max().max() + 1]
    xs = np.array(lims)
    hb = ax_xy.hexbin(judged_all["activeSetpoint"], judged_all["readback"], gridsize=70, bins="log",
                      mincnt=1, cmap="Greys", extent=(*lims, *lims))
    fig.colorbar(hb, ax=ax_xy, label="Judged commands (log)")
    ax_xy.fill_between(xs, xs - LARGE_MISMATCH_C, xs + LARGE_MISMATCH_C, color="0.5", alpha=0.15,
                       label=f"\u00b1{LARGE_MISMATCH_C} \u00b0C")
    ax_xy.plot(xs, xs, color="0.5", lw=1)

    ax_t.plot(judged_all.index[~judged_all["mismatch"]], judged_all["error"][~judged_all["mismatch"]],
              ".", ms=1, color="0.75", rasterized=True, label="matching")

    for name, color in CHILLER_COLORS.items():
        g = judged_all[judged_all["chiller"] == name]
        if not len(g):
            continue
        m = g[g["mismatch"]]
        ax_xy.plot(m["activeSetpoint"], m["readback"], ".", ms=4, alpha=0.5, color=color, label=name)
        ax_t.plot(m.index, m["error"], ".", ms=3, alpha=0.5, color=color, label=name)

    ax_xy.set_xlim(lims)
    ax_xy.set_ylim(lims)
    ax_xy.set_xlabel("Commanded set point [\u00b0C]")
    ax_xy.set_ylabel("Chiller readback [\u00b0C]")
    ax_xy.set_title("All judged commands (points: mismatching)", loc="left", fontsize=10)
    ax_xy.legend(loc="upper left")

    for sign in (-1, 1):
        ax_t.axhline(sign * LARGE_MISMATCH_C, color="0.4", ls="--", lw=1)
    t_locator = mdates.AutoDateLocator()
    ax_t.xaxis.set_major_locator(t_locator)
    ax_t.xaxis.set_major_formatter(mdates.ConciseDateFormatter(t_locator))
    ax_t.set_xlabel("Time [UTC]")
    ax_t.set_ylabel("Readback \u2212 command [\u00b0C]")
    ax_t.set_title("Error through time (dashed: large-mismatch threshold)", loc="left", fontsize=10)
    ax_t.legend(loc="lower right", markerscale=4)

    for a in (ax_xy, ax_t):
        a.grid(True, alpha=0.3)
    fig.suptitle(f"Chiller readback vs command \u2014 day_obs {start_day_obs}\u2013{end_day_obs}")
    fig.tight_layout()
    fig.savefig(f"plots/{stem}_mismatch_size.png", dpi=150, bbox_inches="tight")
    show_figure(fig)
else:
    print("No judged commands to plot.")

## Summary by month

Per chiller and calendar month: how many new commands there were and how many were answered, the median
latency, and the hours (and share of judged time) spent mismatched, for all mismatches and for large ones.
A month where `answered_pct` drops or `large_hours` jumps is worth checking against the events table.

In [ ]:
parts = []
if len(latency):
    lat = latency.assign(month=latency["time"].dt.to_period("M"))
    lat_ok = lat[~lat["status"].isin(["already_matched", "chiller_off"])]
    parts.append(lat_ok.groupby(["chiller", "month"]).agg(
        new_commands=("status", "size"),
        answered_pct=("status", lambda s: 100 * (s == "responded").mean()),
        median_latency_s=("latency_s", "median"),
    ))

by_month = judged_all.assign(
    month=judged_all.index.to_period("M"),
    mismatch_min=judged_all["interval_min"] * judged_all["mismatch"],
    large_min=judged_all["interval_min"] * judged_all["large"],
)
month_sums = by_month.groupby(["chiller", "month"])[["interval_min", "mismatch_min", "large_min"]].sum()
parts.append(pd.DataFrame({
    "mismatch_hours": month_sums["mismatch_min"] / 60,
    "mismatch_pct": 100 * month_sums["mismatch_min"] / month_sums["interval_min"],
    "large_hours": month_sums["large_min"] / 60,
    "large_pct": 100 * month_sums["large_min"] / month_sums["interval_min"],
}))

monthly = pd.concat(parts, axis=1)
display(monthly.round(2))

## All mismatch periods

Every run of consecutive mismatching commands, small ones included, largest error first. `mismatch_min`
is the time actually spent mismatched. `off_during` is True when the chiller was switched off somewhere
inside the period. The large periods are the same rows with `max_abs_error_c ≥ LARGE_MISMATCH_C`.

In [ ]:
if len(periods):
    display(periods.sort_values("max_abs_error_c", ascending=False).reset_index(drop=True))
else:
    print("No mismatch periods in this range.")